<a href="https://colab.research.google.com/github/Metanat-Saadat-Rouhi/mnist-cnn-pytorch/blob/main/mnist_cnn_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
# dataset is for MNIST
# transform is to convert images into pytorch format
from torch.utils.data import DataLoader

In [2]:
### load data ###
# convert into a pytorch tensor(with values scaled to the range 0, 1)
transform = transforms.ToTensor()
# after conversion each image become (1, 28, 28)
#(number of channels, height, weight)

train_data = datasets.MNIST(root="data", train=True, transform=transform, download=True)
test_data = datasets.MNIST(root="data", train=False, transform=transform)
# give me the MNIST dataset. (a built-in dataset of handwritten digits)
# root="data" - create a folder name data, and inside it save the MNIST dataset
# train=True - load the training portion of MNIST files
# MNIST is split to training set(60,000 images) and test set(10,000 images)
# transform=transform says befor giving me an image,apply TpTensor to it
# download=Trues - if the dataset is not already on my computer download it
# test_data loads test dataset
# train=False - load the test split not the training split

print("Number of training images", len(train_data))
print("Number of test images:", len(test_data))


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 478kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.4MB/s]

Number of training images 60000
Number of test images: 10000


In [3]:
# what an item in train_data looks like:
image, label = train_data[0]
print("Single image shape:", image.shape)
print("Single label", label)

Single image shape: torch.Size([1, 28, 28])
Single label 5


In [4]:
# create dataloader
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)
# now instead of getting one image at a time, u get batches
# batch_size=64 - give 64 image per batch
# shuffle=True - mix the order of training data every epoch(so model learns more rebustly)
# in creating test dataloader we ususally do not need to shuffle

In [5]:
# what a batch from train_loader look like:
for images, labels in train_loader:
    print("batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    break

batch image shape: torch.Size([64, 1, 28, 28])
Batch label shape: torch.Size([64])


In [14]:
### class CNN ###
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # input shape = (64, 1, 28, 28)
        self.conv = nn.Conv2d(1, 8, kernel_size=3)
        # input channel=1, output channel=8, filter size=3*3
        # after this: (64, 8, 26, 26)
        # why 26? new_size = old_size - kernel + 1 --- 28-3+1=26
        # now the image has 8 feature maps. each one highlights sth different

        self.pool = nn.MaxPool2d(2)
        # Takes a 2×2 block and keeps the maximum value
        # [1 5]
        # [2 3] → 5
        # why? reduces size, keeps important info, makes model faster
        # after pool: (64, 8, 13, 13) - size it cut in half

        self.fc = nn.Linear(8 * 13 * 13, 10)
        # fully connected layer
        # input 1352 features, output 10 numbers
        # why 10? bc MNIST  has 10 digits(from 0 to 9)
        # [score0, score1, ..., score9] - highest score = predicted digit

    def forward(self, x):
        x = self.conv(x) # detect features
        # print(x.shape)
        x = torch.relu(x) # add non-linearity
        # negative → 0, positive → stays
        # print(x.shape)
        x = self.pool(x) # shrink image
        # print(x.shape)

        x = x.view(x.size(0), -1) # flatten
        # truns (64, 8, 13, 13) into (64, 1352)
        # why 1352? 8 × 13 × 13 = 1352
        # why? Linear layers expect vectors, not images

        x = self.fc(x) # classify
        return x

### setup training ###
model = CNN()

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

### train ###
for epoch in range(3):
    for images, labels in train_loader:

        pred = model(images)
        loss = loss_fn(pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.2453
Epoch 1, Loss: 0.1471
Epoch 2, Loss: 0.1738


In [15]:
### test ####
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", correct / total)

Accuracy: 0.9688


In [8]:
# Input:        (64, 1, 28, 28)
# Conv:         (64, 8, 26, 26)
# ReLU:         (64, 8, 26, 26)
# Pool:         (64, 8, 13, 13)
# Flatten:      (64, 1352)
# Linear:       (64, 10)
# final output: 10 numbers
# [0.1, 0.2, 5.3, 0.1, ..., 0.05] highest value → predicted class

